# 1. 线程

引言：平时写的 Python 程序大多是“单任务模式”——做完一件事再做另一件，就像先从头到尾唱完一首歌，再开始跳舞，全程只能专注于一件事，效率偏低。而线程，正是让程序拥有“一心多用”能力的工具：它能让程序同时推进多个任务，就像人可以边唱歌边跳舞，多个操作并行进行，充分调动 CPU 资源。学会线程，能摆脱单任务的效率束缚，轻松应对多 IO 操作、多请求处理等场景，大幅提升程序运行效率。


## 1. 多任务：同一时间做多个事

### 1.1. 什么是多任务？

多任务就是同一时间段内执行多个任务，比如：

- 开车时，手控方向盘、脚控油门 / 刹车，同时进行；
- 电脑上，边听音乐、边写代码、边下载文件，并行执行。

示例：单任务 vs 多任务（代码对比）

In [2]:
import time
def sing():
    print("Singing a song...")
    time.sleep(2)
    print("Finished singing.")

def dance():
    print("Dancing...")
    time.sleep(2)
    print("Finished dancing.")

# 先唱歌再跳舞
print("单任务执行（耗时4秒）")
start=time.time()
sing()
dance()
print("单任务总耗时：",time.time()-start,"秒")


单任务执行（耗时4秒）
Singing a song...
Finished singing.
Dancing...
Finished dancing.
单任务总耗时： 4.001613616943359 秒


## 2. 认识线程：多任务的“最小执行单元”

### 2.1. 核心概念

- 进程：打开一个程序就会创建一个进程（比如打开微信），是操作系统分配资源的基本单位；
- 线程：进程内的“最小执行单元”（比如微信的聊天、发文件功能），是 CPU 调度的基本单位；
- 关系：一个进程默认有 1 个主线程，还能创建多个子线程，线程依附于进程存在，没有进程就没有线程。

### 2.2. 线程的核心优势

- 轻量级：创建和切换线程的成本低，比进程更高效；
- 共享资源：同一进程内的线程共享全局变量、文件等资源，通信方便；
- 并发执行：多个线程同时执行，充分利用 CPU 多核资源。

---

## 3. 线程基础：创建与启动线程

Python 通过 `threading` 模块实现多线程，核心是 `Thread` 类。

步骤超简单：导入模块 → 创建线程对象 → 启动线程。

### 3.1. 基础用法

In [3]:
import time
def sing():
    print("Singing a song...")
    time.sleep(2)
    print("Finished singing.")

def dance():
    print("Dancing...")
    time.sleep(2)
    print("Finished dancing.")

import threading
 
# 程序有的默认是主程序，自己创建的是子线程
if __name__ == "__main__":
    print("多任务执行（耗时2秒）")
    start=time.time()
    t1 = threading.Thread(target=sing)
    t2 = threading.Thread(target=dance)
    t1.start()
    t2.start()
    t1.join()           # 等待t1线程结束
    t2.join()           # 等待t2线程结束，保证主线程最后结束
    print("多任务总耗时：", time.time() - start, "秒")

多任务执行（耗时2秒）
Singing a song...
Dancing...
Finished singing.
Finished dancing.
多任务总耗时： 2.0015110969543457 秒


### 3.2 进阶用法
如果任务函数需要参数，用`args`（元组）或`kwargs`（字典）传参，两种方式按需选择。

In [4]:
# 带参数的任务函数
def sing(song_name):
    print(f"Singing {song_name}...")
    time.sleep(2)
    print(f"Finished singing {song_name}.")

def dance(dance_style,duration):
    print(f"Dancing {dance_style}...")
    time.sleep(duration)
    print(f"Finished dancing {dance_style}.")

if __name__ == "__main__":
    print("多任务执行（耗时2秒）")
    start = time.time()
    t1 = threading.Thread(target=sing, args=("Happy Song",))           # 元组参数一个元素要加逗号
    t2 = threading.Thread(target=dance, kwargs={"dance_style": "Hip Hop", "duration": 2})
    t1.start()
    t2.start()
    t1.join()
    t2.join()
    print("多任务总耗时：", time.time() - start, "秒")

多任务执行（耗时2秒）
Singing Happy Song...
Dancing Hip Hop...
Finished singing Happy Song.
Finished dancing Hip Hop.
多任务总耗时： 2.0022130012512207 秒


### 3.3 关键特性：线程执行是无序的  
线程的执行顺序由CPU调度决定，不是创建顺序，每次运行结果可能不同

In [11]:
def print_num(num):
    time.sleep(0.1)
    print(f"线程{num}:打印数字: {num}")

if __name__ == "__main__":
    print("线程执行顺序")
    for i in range(1,4):
        t = threading.Thread(target=print_num,args=(i,))
        t.start()

线程执行顺序


线程1:打印数字: 1线程2:打印数字: 2
线程3:打印数字: 3



---

## 4. 线程的核心问题：共享资源与资源竞争
### 4.1 线程共享全局资源
同一进程内的线程会共享全局变量、文件等资源，这是线程的优势，但也会引发问题。

In [14]:
# 全局变量（共享资源）
count = 0

def add_count():
    global count
    print(f"线程1：当前count={count}")
    time.sleep(0.1)
    count += 1
    print(f"线程1：count加1后 = {count}")

def sub_count():
    global count
    print(f"线程2：当前count={count}")
    time.sleep(0.1)
    count -= 1
    print(f"线程2：count减1后 = {count}")

if __name__ == "__main__":
    t1 = threading.Thread(target=add_count)
    t2 = threading.Thread(target=sub_count)
    t1.start()
    t2.start()

线程1：当前count=0
线程2：当前count=0


线程2：count减1后 = -1
线程1：count加1后 = 0


In [ ]:
# 全局变量
count =0 
def add_count():
    global count
    for _ in range(10000000):
        count += 1
    print(f"执行后count:{count}")

if __name__ == "__main__":
    # 创建2个子线程同时修改count
    t1 = threading.Thread(target=add_count)
    t2 = threading.Thread(target=add_count)
    t1.start()
    t2.start()

执行后count:19819655
执行后count:20000000


## 5. 线程同步：解决资源竞争的两种方案

线程同步是让多个线程按顺序执行，避免资源竞争，常用两种方式：join（线程等待）和 互斥锁。

### 5.1. 方案 1：join 线程等待

让主线程或其他线程等待目标线程执行完毕后，再继续执行，适合“先执行 A 再执行 B”的场景。  
但是失去了多任务并发的优势，总耗时会变成两个线程的时间之和，适合对执行顺序有要求的场景(先写后读)，不适合追求效率的场景

In [18]:
# 全局变量
count =0 
def add_count():
    global count
    for _ in range(10000000):
        count += 1
    print(f"执行后count:{count}")

if __name__ == "__main__":
    # 创建2个子线程同时修改count
    t1 = threading.Thread(target=add_count)
    t2 = threading.Thread(target=add_count)
    t1.start()
    t1.join()   #等待t1执行完
    t2.start()
    t2.join()

执行后count:10000000
执行后count:20000000


### 5.2. 方案 2：互斥锁（Lock）

互斥锁能保证“同一时间只有一个线程能操作共享资源”，就像公共厕所的门锁——有人用的时候锁上，用完再解锁，其他人才能用。

In [ ]:
# 全局变量
count =0 
lock = threading.Lock()

def add_count():
    global count
    lock.acquire()
    try:
        for _ in range(10000000):
            count += 1
    finally:
        lock.release()      #必须释放锁！！
    print(f"执行后count:{count}")

if __name__ == "__main__":
    # 创建2个子线程同时修改count
    t1 = threading.Thread(target=add_count)
    t2 = threading.Thread(target=add_count)
    t1.start()
    t2.start()

执行后count:10000000
执行后count:20000000


#### 关键注意：避免死锁！

死锁是多线程的致命问题——两个线程互相等待对方释放锁，导致永远阻塞。

#### 避坑指南

- 上锁后必须解锁：用 `try...finally` 确保解锁，避免程序出错导致锁未释放；
- 减少锁的范围：只给“修改共享资源的代码块”上锁，不要给整个函数上锁（影响效率）；
- 避免多把锁嵌套：尽量用一把锁解决问题，多锁嵌套容易导致死锁。

---


## 6. 核心总结
1. 线程核心：实现多任务并发，轻量级、共享资源、CPU 调度执行（无序）。

2. 线程创建：`threading.Thread(target=任务函数, args=元组参数, kwargs=字典参数)`，启动用 `start()`。

3. 关键问题：

   - 共享资源：同一进程的线程共享全局变量，通信方便；
   - 资源竞争：多个线程同时修改共享资源，导致数据错误。

4. 解决方案：

   - `join()`：线程等待，适合“先 A 后 B”的顺序执行；
   - 互斥锁（Lock）：`acquire()` 上锁，`release()` 解锁，确保同一时间只有一个线程操作共享资源。

5. 避坑要点：

   - 线程执行无序，不依赖创建顺序；
   - 互斥锁必须成对使用，避免死锁；
   - 数据修改用锁，数据读取不用锁（只读不会冲突）。